# Azure OpenAI via plain `openai` SDK

Uses the standard `OpenAI` client pointed at your Azure resource's `/openai/v1` endpoint. **No `AzureOpenAI`, no Entra ID, no device code login.** Same code shape as OpenAI direct, OpenRouter, Together, Groq — swap `base_url` to switch providers.

**Sections (in the order you learned them):**
1. Chat Completions — the bread and butter (+ interactive conversation history)
2. Responses API — newer, cleaner, server-side state
3. Streaming
4. **File Search** — managed RAG, zero infrastructure
5. **Web Search** — Bing-grounded with citations
6. **Code Interpreter** — sandboxed Python (with and without file upload)
7. **Custom Function Calling** — your own Python functions as tools
8. **Agentic Search** — agent loop using `read_file` over the same RAG corpus
9. **Text Editing Agent** — list / read / edit tools (a tiny coding agent)
10. **Deep Research Agent** — orchestrator with parallel sub-agent searches and a validator loop

**You only need to fill in TWO things below:**
1. **Your resource name** — visible on the Foundry portal front page, or the part of the portal URL right after `https://` and before the first dot
2. **Your API key** — Foundry portal front page → API Key → click the copy button

In [ ]:
!pip install -q openai

## Setup — fill in the two values, then run the cell

In [ ]:
from openai import OpenAI

# === FILL IN THESE TWO ===
RESOURCE_NAME = "nutjung-nutlc-0792-resource"   # your Foundry resource name
API_KEY       = "paste-your-api-key-here"        # from Foundry portal front page
# =========================

MODEL = "gpt-4o"  # change ONLY if you named your deployment something else

# If you get a DNS error, change "openai" to "cognitiveservices" below
BASE_URL = f"https://{RESOURCE_NAME}.openai.azure.com/openai/v1"

# default_query is required for built-in tools (file_search etc.) on Azure preview
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    default_query={"api-version": "preview"},
)
print(f"Client ready. Endpoint: {BASE_URL}")

## Verify the connection works

One tiny call. If this prints a greeting, everything's wired up correctly.

In [ ]:
ping = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in 5 words."}],
)
print(ping.choices[0].message.content)

## 1. Chat Completions

The classic OpenAI API surface. A list of `messages` (system / user / assistant) goes in, a single response comes out. **Stateless** — the model has no memory between calls. To carry on a conversation, you maintain the message list yourself (see 1b below).

📖 **Docs:** [OpenAI Chat Completions reference](https://platform.openai.com/docs/api-reference/chat) · [Azure quickstart](https://learn.microsoft.com/en-us/azure/ai-services/openai/chatgpt-quickstart)

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "Name three Thai street food dishes."},
    ],
    temperature=0.7,
)
print(resp.choices[0].message.content)

### 1b. Conversation history (interactive while-loop)

Because chat completions are stateless, multi-turn conversations require **you** to keep the history and pass it on every call. The pattern: append the user's new message → call the API → append the assistant's response back into the list → repeat.

Run the cell below and chat with the bot. Type `quit` (or just press enter on an empty line) to exit. Compare this with the Responses API (Section 2) where the server stores history for you — just pass `previous_response_id` and skip all this bookkeeping.

In [ ]:
messages = [
    {"role": "system", "content": "You are a Thai food expert. Keep answers to 1-2 sentences."},
]

print("Chat with the Thai food bot. Type 'quit' (or enter empty) to exit.\n")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ("quit", "exit", ""):
        print("Goodbye!")
        break

    # Append the user message → send → append the reply back into history
    messages.append({"role": "user", "content": user_input})
    r = client.chat.completions.create(model=MODEL, messages=messages)
    reply = r.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})

    print(f"Bot: {reply}\n")

print(f"Total messages stored in history: {len(messages)}")
print("(System + alternating user/assistant pairs)")

## 2. Responses API

Newer, simpler shape. A single `input` field instead of a `messages` array. **Built-in conversation chaining** via `previous_response_id` — the server stores the history for you. Required for the built-in tools (file search, web search, code interpreter) below.

📖 **Docs:** [Azure Responses API guide](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses) · [OpenAI Responses reference](https://platform.openai.com/docs/api-reference/responses)

In [ ]:
r1 = client.responses.create(
    model=MODEL,
    input="What is pad krapow?",
    instructions="Answer in one sentence.",
)
print(r1.output_text)

# Continue the conversation by chaining response IDs (no manual history needed)
r2 = client.responses.create(
    model=MODEL,
    input="What is the typical price in Bangkok?",
    previous_response_id=r1.id,
)
print(r2.output_text)

## 3. Streaming

Same as OpenAI direct. Tokens print as they're generated. Useful for chat UIs where you want the user to see output immediately rather than waiting for the full response. Ask for a longer output (a story, an explanation) so you can actually see the streaming effect.

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": (
            "Write a 200-word short story about a robot learning to make pad thai "
            "for the first time. Include a small mishap."
        )},
    ],
    stream=True,
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

---

# Built-in tools

The Responses API exposes three managed tools that Azure runs server-side. You don't write any tool code — just declare the tool and the model uses it. They cost extra (Bing for web_search, container compute for code_interpreter, vector storage for file_search) but save you from building infrastructure.

All three only work via `client.responses.create(...)`, not `client.chat.completions.create(...)`.

## 4. File Search (managed RAG)

Upload documents → Azure embeds and indexes them in a **vector store** → the model retrieves relevant chunks at query time. Zero-infrastructure RAG: no Pinecone, no Chroma, no chunking code.

📖 **Docs:** [Azure file_search tool reference](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/file-search)

**Pattern:**
1. Upload files via `client.files.create(...)`
2. Create a vector store via `client.vector_stores.create(...)` and pass the file IDs
3. Poll until status is `completed`
4. Pass the vector store ID to `tools=[{"type": "file_search", "vector_store_ids": [...]}]`

We'll use the 50 Contoso product markdown files in `./products/` as our corpus, and the eval questions in `rag-questions.csv` as test queries.

### Step 1 — Upload files and build the vector store

Run this once per session. Takes ~30 seconds for 50 files.

In [ ]:
import glob, time

# Upload every product markdown file
product_files = sorted(glob.glob("products/*.md"))
print(f"Uploading {len(product_files)} files...")

file_ids = []
for path in product_files:
    with open(path, "rb") as f:
        up = client.files.create(file=f, purpose="assistants")
        file_ids.append(up.id)
print(f"Uploaded {len(file_ids)} files.")

# Create the vector store
vector_store = client.vector_stores.create(
    name="contoso-product-catalog",
    file_ids=file_ids,
)
print(f"Vector store created: {vector_store.id} (status: {vector_store.status})")

# Poll until indexing is complete
while vector_store.status != "completed":
    time.sleep(2)
    vector_store = client.vector_stores.retrieve(vector_store.id)
    print(f"  ...{vector_store.status}")

print("Ready!")

### Example 4a — Simple lookup (S1 from rag-questions.csv)

*Golden answer: yes, 72Wh < 100Wh IATA limit, carry-on only.*

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
    input="Is the PowerVault portable charger allowed on airplanes?",
)
print(r.output_text)

### Example 4b — Product variant disambiguation (P1)

Two SKUs share a brand name. The model has to retrieve both and distinguish them.

*Golden answer: FitPulse Ultra has SpO2, FitPulse Lite does not.*

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
    input="Does the Contoso FitPulse fitness tracker have blood oxygen monitoring?",
)
print(r.output_text)

### Example 4c — Hallucination edge case (E6)

The product doesn't exist in the catalog. A good RAG should say so, NOT fabricate.

*Golden answer: SmartLock Pro is not a Contoso product. (Acceptable to mention WatchGuard doorbell as the closest existing security product.)*

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
    input="Tell me about the Contoso SmartLock Pro door lock.",
)
print(r.output_text)

## 5. Web Search

Lets the model retrieve and ground its answers with real-time information from the public web, with inline citations. Powered by Bing Grounding under the hood. Costs ~$30/1000 queries (separate from token costs).

📖 **Docs:** [Azure Web Search with Responses API](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/web-search)

**Three modes:** *without reasoning* (gpt-4o, fast), *agentic* (o3, plans multiple searches), *deep research* (`o3-deep-research`, multi-step investigation, can take minutes).

### Example 5a — Basic web search

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{"type": "web_search"}],
    input="What is the latest stable Python version released? Cite your source.",
)
print(r.output_text)

### Example 5b — Region-aware search

Pass `user_location` to bias results toward a country. Useful for local news, prices, regulations.

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{
        "type": "web_search",
        "user_location": {"type": "approximate", "country": "TH"},
    }],
    input="What's a popular tourist destination in Thailand right now?",
)
print(r.output_text)

### Example 5c — Domain-restricted search

Limit results to specific trusted domains (up to 100). Great for medical, legal, or research use cases where you only want authoritative sources.

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{
        "type": "web_search",
        "filters": {
            "allowed_domains": ["docs.python.org", "peps.python.org"],
        },
    }],
    input="What does PEP 8 say about line length?",
)
print(r.output_text)

## 6. Code Interpreter

Spins up a sandboxed Python container, lets the model write and run code, returns the result. Perfect for math, data analysis, plotting, file format conversion. Two flavors: with or without an attached file.

📖 **Docs:** [Azure Responses API — Code Interpreter section](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses)

Containers expire after 20 minutes of inactivity, so re-running cells in a long session may spin up a fresh one.

**The helper below** prints both the **executed Python code** and the model's natural-language answer. Without it you'd only see the answer — and the whole point of code interpreter is being able to inspect what code actually ran.

In [ ]:
def print_with_code(r):
    """Pretty-print a code_interpreter response: shows both the code AND the answer."""
    DIM, CYAN, RESET = "\033[2m", "\033[96m", "\033[0m"
    for item in r.output:
        if item.type == "code_interpreter_call":
            print(f"{DIM}{'─' * 60}{RESET}")
            print(f"{CYAN}▼ Code executed by the model:{RESET}")
            print(f"{DIM}{item.code}{RESET}")
            print(f"{DIM}{'─' * 60}{RESET}\n")
        elif item.type == "message":
            for c in item.content:
                if c.type == "output_text":
                    print(c.text)

### 6a. Without a file — pure compute

The model writes Python, runs it in the container, returns the answer.

#### Example 6a-i — Big integer math

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
    input="Compute 100 factorial. Then count how many trailing zeros it has. Use Python code, do not just answer.",
)
print_with_code(r)

#### Example 6a-ii — ASCII art via code

The prompt explicitly says "use Python code" because gpt-4o is smart enough to draw small pyramids in its head — we want to force the container to actually execute code so you can see it in the output.

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
    input=(
        "Use Python code to generate and print a Christmas tree of stars 10 rows tall, centered. "
        "You MUST run code, do not just write the answer directly."
    ),
)
print_with_code(r)

### 6b. With a file upload — data analysis

Upload a CSV/JSON/XLSX, attach its file ID to the container, and the model can `pd.read_csv(...)` it directly. This is the killer feature — you basically get a sandboxed data analyst.

In [ ]:
# Upload the CSV once and reuse the file_id
with open("thai_street_food.csv", "rb") as f:
    csv_file = client.files.create(file=f, purpose="assistants")
print(f"Uploaded: {csv_file.id}")

#### Example 6b-i — Simple aggregation

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{
        "type": "code_interpreter",
        "container": {"type": "auto", "file_ids": [csv_file.id]},
    }],
    input="What is the average price of dishes in the noodles category?",
)
print_with_code(r)

#### Example 6b-ii — Group-by analysis

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{
        "type": "code_interpreter",
        "container": {"type": "auto", "file_ids": [csv_file.id]},
    }],
    input="How many dishes come from each region? Which region has the most?",
)
print_with_code(r)

#### Example 6b-iii — Multi-step analytical question

In [ ]:
r = client.responses.create(
    model=MODEL,
    tools=[{
        "type": "code_interpreter",
        "container": {"type": "auto", "file_ids": [csv_file.id]},
    }],
    input=(
        "Find the dish with the highest monthly sales. "
        "Then compute its total annual revenue (price × monthly_sales × 12)."
    ),
)
print_with_code(r)

## 7. Custom Function Calling

You declare a Python function as a tool, the model decides to call it, you run it, feed the result back. This is how you let the model touch external systems (databases, APIs, calculators) when no built-in tool fits.

In [ ]:
import json

def get_exchange_rate(from_currency: str, to_currency: str) -> str:
    rates = {"USD": 35.5, "EUR": 38.2, "JPY": 0.24, "THB": 1.0}
    rate = rates[from_currency] / rates[to_currency]
    return f"1 {from_currency} = {rate:.4f} {to_currency}"

tools = [{
    "type": "function",
    "function": {
        "name": "get_exchange_rate",
        "description": "Get exchange rate between two currencies.",
        "parameters": {
            "type": "object",
            "properties": {
                "from_currency": {"type": "string"},
                "to_currency":   {"type": "string"},
            },
            "required": ["from_currency", "to_currency"],
        },
    },
}]

messages = [{"role": "user", "content": "How much is 100 USD in THB?"}]

resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
msg = resp.choices[0].message
messages.append(msg)

for call in (msg.tool_calls or []):
    args = json.loads(call.function.arguments)
    result = get_exchange_rate(**args)
    messages.append({
        "role": "tool",
        "tool_call_id": call.id,
        "content": result,
    })

final = client.chat.completions.create(model=MODEL, messages=messages)
print(final.choices[0].message.content)

---

# Agent patterns

The next three sections build progressively more capable agents using only the custom function-calling pattern from Section 7. They share one helper (`run_agent`) — a generic loop that keeps calling the model until it stops requesting tools.

**Why this matters:** built-in tools are convenient but opaque. When you write your own tools you control exactly what the agent can touch — the same approach used by Claude Code, Cursor, and most production agentic systems.

### The generic agent loop (used by Sections 8, 9, 10)

Run this once. The three following sections all reuse it. The verbose output is colored: **green arrow** = a tool call, **bold name** = the function being invoked, **dim** = the arguments (truncated for readability).

In [ ]:
# ANSI color codes — render correctly in Jupyter notebook output
_GREEN = "\033[92m"
_CYAN  = "\033[96m"
_DIM   = "\033[2m"
_BOLD  = "\033[1m"
_RESET = "\033[0m"

def _format_args(args_json: str, max_value_len: int = 70) -> str:
    """Render tool-call arguments as compact key=value, truncating long values."""
    try:
        args = json.loads(args_json)
    except Exception:
        return args_json[:max_value_len]
    parts = []
    for k, v in args.items():
        v_str = str(v).replace("\n", " ")
        if len(v_str) > max_value_len:
            v_str = v_str[: max_value_len - 3] + "..."
        parts.append(f"{k}={v_str!r}")
    return ", ".join(parts)

def run_agent(messages, tools, tool_map, max_iters=15, verbose=True):
    """Generic agent loop. Keeps calling the model until it stops requesting tools.
    
    - messages: a list of chat messages (will be mutated)
    - tools:    OpenAI-format tool definitions
    - tool_map: dict mapping tool name -> Python callable
    """
    for i in range(max_iters):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            return msg.content  # done

        if verbose:
            print(f"{_DIM}[iter {i}]{_RESET} {_CYAN}tool calls:{_RESET}")
            for c in msg.tool_calls:
                args_str = _format_args(c.function.arguments)
                print(f"  {_GREEN}→{_RESET} {_BOLD}{c.function.name}{_RESET}({_DIM}{args_str}{_RESET})")

        for call in msg.tool_calls:
            fn = tool_map[call.function.name]
            args = json.loads(call.function.arguments)
            try:
                result = fn(**args)
            except Exception as e:
                result = f"ERROR: {e}"
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })
    return "[max iterations reached]"

## 8. Agentic Search (custom RAG)

Same product corpus as Section 4, but instead of managed RAG with embeddings, we give the agent a single `read_product_file` tool and put the **list of all filenames in the system prompt**. The model picks which files to read based on the question.

**Tradeoffs vs. file_search:**
- ✅ No vector store needed, no embedding costs
- ✅ Model sees full file content, not just retrieved chunks
- ✅ Easy to add other tools (search, web lookup) to the same agent
- ❌ System prompt grows with corpus size — fine for 50 files, breaks at 5,000
- ❌ Model may need multiple iterations if it picks the wrong file first

In [ ]:
import os

PRODUCTS_DIR = "products"
PRODUCT_FILE_LIST = sorted(os.listdir(PRODUCTS_DIR))

def read_product_file(filename: str) -> str:
    """Read the contents of a product markdown file from the catalog."""
    path = os.path.join(PRODUCTS_DIR, filename)
    if not os.path.exists(path):
        return f"FILE NOT FOUND: {filename}"
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

agentic_search_tools = [{
    "type": "function",
    "function": {
        "name": "read_product_file",
        "description": "Read the contents of a product markdown file from the catalog.",
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {"type": "string", "description": "e.g. 'portable-charger.md'"},
            },
            "required": ["filename"],
        },
    },
}]

agentic_search_tool_map = {"read_product_file": read_product_file}

AGENTIC_SEARCH_PROMPT = (
    "You answer questions about Contoso products. Use the read_product_file tool "
    "to read the file most relevant to the question, then answer based on its content. "
    "Do not invent facts. If multiple files might be relevant, read them all.\n\n"
    f"Available files:\n{', '.join(PRODUCT_FILE_LIST)}"
)

def ask_agentic(question: str) -> str:
    messages = [
        {"role": "system", "content": AGENTIC_SEARCH_PROMPT},
        {"role": "user", "content": question},
    ]
    return run_agent(messages, agentic_search_tools, agentic_search_tool_map)

### Example 8a — Single-file lookup

In [ ]:
print(ask_agentic("Is the PowerVault portable charger allowed on airplanes?"))

### Example 8b — Cross-document reasoning (model has to read 2 files)

In [ ]:
print(ask_agentic("Does the Contoso FitPulse fitness tracker have blood oxygen monitoring?"))

## 9. Text Editing Agent

A baby coding agent. Three tools — `list_files`, `read_file`, `edit_file` — over a sandboxed directory. The model decides what to read, what to change, and how to fix it.

**This is the core loop behind tools like Claude Code, Cursor, Aider, and Continue.** They have more tools (bash, grep, write_file, etc.) but the pattern is identical.

### Step 1 — Create the sandbox (idempotent)

Re-running this cell resets the sandbox to a known state — useful between demos.

In [ ]:
import os

SANDBOX = "editor_sandbox"
os.makedirs(SANDBOX, exist_ok=True)

with open(os.path.join(SANDBOX, "notes.md"), "w", encoding="utf-8") as f:
    f.write("# Meeting Notes\n\nThe team will recieve new laptops on Friday.\n")

with open(os.path.join(SANDBOX, "config.json"), "w", encoding="utf-8") as f:
    f.write('{"port": 3000, "host": "localhost", "debug": false}\n')

with open(os.path.join(SANDBOX, "script.py"), "w", encoding="utf-8") as f:
    f.write("def old_function_name():\n    return 42\n\nprint(old_function_name())\n")

print("Sandbox files:", os.listdir(SANDBOX))

### Step 2 — Tool definitions

In [ ]:
def list_files(directory: str = ".") -> str:
    target = SANDBOX if directory == "." else os.path.join(SANDBOX, directory)
    if not os.path.exists(target):
        return f"DIRECTORY NOT FOUND: {directory}"
    return "\n".join(sorted(os.listdir(target)))

def read_file(path: str) -> str:
    full = os.path.join(SANDBOX, path)
    if not os.path.exists(full):
        return f"FILE NOT FOUND: {path}"
    with open(full, "r", encoding="utf-8") as f:
        return f.read()

def edit_file(path: str, old_string: str, new_string: str) -> str:
    full = os.path.join(SANDBOX, path)
    if not os.path.exists(full):
        return f"FILE NOT FOUND: {path}"
    with open(full, "r", encoding="utf-8") as f:
        content = f.read()
    if old_string not in content:
        return f"OLD STRING NOT FOUND in {path}"
    with open(full, "w", encoding="utf-8") as f:
        f.write(content.replace(old_string, new_string))
    return f"OK: edited {path}"

editor_tools = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files in a directory (default: sandbox root).",
            "parameters": {
                "type": "object",
                "properties": {"directory": {"type": "string", "default": "."}},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a file.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "edit_file",
            "description": "Replace old_string with new_string in a file. The old_string must match exactly and must appear exactly once.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "old_string": {"type": "string"},
                    "new_string": {"type": "string"},
                },
                "required": ["path", "old_string", "new_string"],
            },
        },
    },
]

editor_tool_map = {
    "list_files": list_files,
    "read_file": read_file,
    "edit_file": edit_file,
}

EDITOR_PROMPT = (
    "You are a text-editing agent operating in a sandboxed directory. "
    "Use list_files to discover what's there, read_file to inspect, edit_file to modify. "
    "Always verify your edits by reading the file again afterwards."
)

def ask_editor(task: str) -> str:
    messages = [
        {"role": "system", "content": EDITOR_PROMPT},
        {"role": "user", "content": task},
    ]
    return run_agent(messages, editor_tools, editor_tool_map)

### Example 9a — Fix a typo

The agent should: list → read notes.md → spot 'recieve' → edit_file.

In [ ]:
print(ask_editor("There's a typo somewhere in the notes file. Find it and fix it."))
print("\n--- File after edit ---")
print(open(os.path.join(SANDBOX, "notes.md"), encoding="utf-8").read())

### Example 9b — Update a config value

In [ ]:
print(ask_editor("Change the port in config.json from 3000 to 8080."))
print("\n--- File after edit ---")
print(open(os.path.join(SANDBOX, "config.json"), encoding="utf-8").read())

### Example 9c — Rename a function across a file

In [ ]:
print(ask_editor("In script.py, rename `old_function_name` to `compute_answer`. Update both the definition and the call site."))
print("\n--- File after edit ---")
print(open(os.path.join(SANDBOX, "script.py"), encoding="utf-8").read())

## 10. Deep Research Agent (multi-agent orchestrator)

The most sophisticated pattern in this notebook. An **orchestrator** agent coordinates three **sub-agents** (each is just a Python function that calls the LLM) to research a question:

- `search_subagent(query, original_question)` — runs one web_search call AND **compresses the result** to only the bullet points relevant to the original question
- `validate(question, notes)` — checks if notes are sufficient (returns COMPLETE/INCOMPLETE)
- `write_report(question, notes)` — synthesizes the final report

**Why the compression matters:** raw web search results are 1k-3k tokens each, with citations, navigation text, and unrelated links. With 3-5 parallel searches per round and possibly 2 rounds, raw results blow the context window fast. The sub-agent runs a second LLM call to extract only the 3-5 facts that answer the original question, then returns those bullet points. The orchestrator stays lean.

**The interesting bit:** the orchestrator can call `search_subagent` **in parallel** in a single turn, fanning out 3-5 different angles on the question simultaneously. We verified earlier that gpt-4o on Azure issues parallel tool calls correctly.

**Flow:**
1. Orchestrator plans 3-5 search queries → calls `search_subagent` × N in parallel
2. Each sub-agent searches THEN compresses to relevant bullets only
3. Orchestrator receives compressed results → calls `validate`
4. If INCOMPLETE → plans 2-3 more searches → repeats validate
5. When COMPLETE → calls `write_report` → returns the report

### Step 1 — Sub-agent functions

In [ ]:
def search_subagent(query: str, original_question: str) -> str:
    """Sub-agent: web search → compress → return only relevant facts as bullets.
    
    Two LLM calls per invocation:
    1. Web search with the specific query (returns raw results, often 2k+ tokens)
    2. Compression pass that extracts ONLY bullets relevant to the original question
    
    This keeps the orchestrator's context lean even with 5+ parallel searches.
    """
    # Step 1: raw web search
    raw = client.responses.create(
        model=MODEL,
        tools=[{"type": "web_search"}],
        input=query,
        instructions="Find the most relevant facts. Cite sources inline.",
    ).output_text

    # Step 2: compress to ONLY what's relevant to the original research question
    compressed = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "Extract ONLY the facts from the search results that help answer the original "
                "research question. Output 3-5 bullet points maximum. Each bullet: one fact + "
                "its source URL. Discard general background, navigation text, citations to "
                "unrelated topics, and filler. If nothing in the results is relevant, return "
                "a single bullet saying 'No relevant facts found for this query.'"
            )},
            {"role": "user", "content": (
                f"ORIGINAL RESEARCH QUESTION: {original_question}\n\n"
                f"SEARCH QUERY USED: {query}\n\n"
                f"RAW SEARCH RESULTS:\n{raw}"
            )},
        ],
    ).choices[0].message.content
    return compressed

def validate(question: str, notes: str) -> str:
    """Sub-agent: checks if notes are sufficient. Returns 'COMPLETE' or 'INCOMPLETE'."""
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "Decide if the notes are sufficient to fully answer the question. "
                "Reply with EXACTLY ONE WORD: COMPLETE or INCOMPLETE. No other text."
            )},
            {"role": "user", "content": f"Question: {question}\n\nNotes:\n{notes}"},
        ],
    )
    return r.choices[0].message.content.strip()

def write_report(question: str, notes: str) -> str:
    """Sub-agent: writes the final research report from gathered notes."""
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "Write a 3-paragraph research report answering the question. "
                "Cite sources inline using the URLs in the notes. Be specific and factual."
            )},
            {"role": "user", "content": f"Question: {question}\n\nNotes:\n{notes}"},
        ],
    )
    return r.choices[0].message.content

### Step 2 — Orchestrator wiring

In [ ]:
orchestrator_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_subagent",
            "description": (
                "Run ONE web search query AND compress the results to only the facts relevant "
                "to the original research question. Returns 3-5 bullet points with source URLs. "
                "Call this multiple times IN PARALLEL (in the same turn) for different angles."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The specific web search query"},
                    "original_question": {
                        "type": "string",
                        "description": "The user's original research question — used to filter what's relevant",
                    },
                },
                "required": ["query", "original_question"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "validate",
            "description": "Check if accumulated notes are sufficient. Returns COMPLETE or INCOMPLETE.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                    "notes": {"type": "string", "description": "All compressed findings concatenated."},
                },
                "required": ["question", "notes"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_report",
            "description": "Write the final research report. Call this ONCE after validate returns COMPLETE.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                    "notes": {"type": "string"},
                },
                "required": ["question", "notes"],
            },
        },
    },
]

orchestrator_tool_map = {
    "search_subagent": search_subagent,
    "validate": validate,
    "write_report": write_report,
}

ORCHESTRATOR_PROMPT = """You are a research orchestrator. Answer the user's question by coordinating sub-agents.

PROCESS (follow EXACTLY):
1. Plan 3-5 specific search queries covering different angles of the question.
2. Call `search_subagent` IN PARALLEL for ALL of them in the SAME turn.
   - Each call: pass the specific `query` AND the user's `original_question` (so the sub-agent compresses correctly).
3. After receiving the compressed bullet points from all searches, compile them into a single notes string.
4. Call `validate` with the question and the notes.
5. If validate returns INCOMPLETE: plan 2-3 MORE specific queries, call search_subagent in parallel (still passing original_question), then validate again.
6. When validate returns COMPLETE: call `write_report` with the question and notes.
7. Return the report from write_report verbatim.

RULES:
- Never write your own answer. Always go through the sub-agents.
- Always call search_subagent in parallel (not one at a time) for efficiency.
- Always pass `original_question` to search_subagent so it can filter context properly.
- Stop after at most 2 validation rounds to avoid infinite loops."""

def deep_research(question: str) -> str:
    messages = [
        {"role": "system", "content": ORCHESTRATOR_PROMPT},
        {"role": "user", "content": question},
    ]
    return run_agent(messages, orchestrator_tools, orchestrator_tool_map, max_iters=15)

### Example 10 — Run a full research query

This will take ~30-60 seconds because it's running multiple web searches in parallel, compressing each one, then validation, then report writing. Watch the colored `[iter N] tool calls:` lines to see the orchestration unfold.

In [ ]:
report = deep_research(
    "What are the latest advances in fusion energy as of 2026? "
    "Include specific milestones, key projects, and remaining challenges."
)
print("\n========== FINAL REPORT ==========\n")
print(report)

## Notes & gotchas

- **Free tier rate limits:** ~1 req/sec on F0. Add `time.sleep(1)` between calls in tight loops.
- **`DeploymentNotFound`?** Your `MODEL` string doesn't match the exact deployment name in the portal. Check Foundry → Models + endpoints.
- **DNS / connection error?** Your resource is on the other Azure DNS suffix. Change `openai.azure.com` to `cognitiveservices.azure.com` in `BASE_URL`.
- **Tool costs (these are NOT free):**
  - `web_search` → ~$30 per 1k Bing queries
  - `code_interpreter` → small per-container compute charge
  - `file_search` → vector store storage + retrieval token costs
  - The deep research agent fans out to many parallel web_search calls — costs add up fast on long questions
- **`web_search` blocked?** Subscription admin can disable it via `az feature register --name OpenAI.BlockedTools.web_search`.
- **Cleanup:** `client.vector_stores.delete(id)` and `client.files.delete(id)` if you're stacking up test artifacts.
- **Switching providers later:** since this uses the standard `OpenAI` client, you can swap `base_url` to OpenAI direct, OpenRouter, Together, Groq, etc. Custom function calling and streaming are universal; built-in tools (file_search, web_search, code_interpreter) are Azure/OpenAI-specific and won't work on other providers.